# Hoá's Midnight Escape: Dueling Double Deep Q-Network (D3QN)
### Complete End-to-End Reinforcement Learning on Google Colab & Kaggle

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Quanninh/Mario_Agent/blob/DQN/notebooks/mario_dqn_colab_kaggle.ipynb)

This notebook implements and trains a **Dueling Double Deep Q-Network (D3QN)** to master all 8 progressive platformer levels of **Hoá's Midnight Escape**.

---

### Key Technical Components:
1. **Continuous Vector Feature Extractor**: Replaces the discrete tabular binning with a normalized 22-dimensional continuous state representation.
2. **Dueling Network Architecture**: Decomposes $Q(s, a)$ into a state-value stream $V(s)$ and advantage stream $A(s, a)$ to stabilize value estimation.
3. **Double Q-Target Optimization**: Decouples action selection from action evaluation to eliminate positive maximization bias.
4. **Experience Replay Buffer**: Breaks temporal correlation with uniform i.i.d. batch sampling.
5. **Polyak Soft Target Updates ($	au = 0.005$)**: Ensures smooth target network tracking.
6. **Curriculum Transfer (Levels 1 $\to$ 8)**: Incrementally trains the neural policy through progressive graveyard challenges, culminating in **Level 8: The Celestial Purgatory** (7,150 px marathon).
7. **In-Notebook Gameplay GIF Recorder**: Off-screen Pygame frame capture to watch your trained agent play directly inside the notebook.


## 1. Environment Setup & Dependency Installation
Configure headless Pygame (using the `"dummy"` video driver for Linux servers) and import/clone the environment repository.


In [ ]:
import os
import sys
from pathlib import Path

# 1. Configure headless Pygame driver (crucial for Colab / Kaggle without X11 display)
os.environ["SDL_VIDEODRIVER"] = "dummy"

# 2. Detect runtime environment (Google Colab vs. Kaggle vs. Local)
is_colab = "google.colab" in sys.modules
is_kaggle = os.path.exists("/kaggle/working")

print(f"Runtime Environment: Colab={is_colab}, Kaggle={is_kaggle}, Local={not is_colab and not is_kaggle}")

# 3. Clone repository if not present locally
repo_dir = Path("Mario_Agent")
if not Path("env").exists() and not (repo_dir / "env").exists():
    print("[INFO] Cloning repository from GitHub...")
    !git clone https://github.com/Quanninh/Mario_Agent.git
    sys.path.append(str(Path("Mario_Agent").resolve()))
elif (repo_dir / "env").exists():
    sys.path.append(str(repo_dir.resolve()))
else:
    sys.path.append(str(Path(".").resolve()))

# 4. Install required packages
!pip install -q pygame torch matplotlib imageio pandas tqdm

import pygame
pygame.init()
print("[OK] Pygame initialized in headless mode successfully!")


## 2. Hardware Acceleration Check
Detect and configure CUDA GPU, Apple Silicon MPS, or CPU.


In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"[ACCELERATION] GPU Detected: {gpu_name} ({gpu_mem:.2f} GB VRAM)")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("[ACCELERATION] Apple Silicon MPS Detected!")
else:
    device = torch.device("cpu")
    print("[ACCELERATION] Running on CPU.")

print(f"PyTorch Version: {torch.__version__} | Active Device: {device}")


## 3. Continuous Feature State Extractor (63-Dimensional Vector)
Instead of coarse tabular discrete bins (which suffer from perceptual aliasing and state space explosion), we extract a continuous normalized 63-dimensional feature vector $\mathbf{s} \in \mathbb{R}^{63}$.

This extractor is synchronized with all `RawState` Section 7 DQN-ready kinematics:
- **Obstacles**: distance, height, relative vertical offset (`obstacle_dy`), and width.
- **Pits & Fire**: distance, width, flame state, fire cycle, countdown, and continuous $(\sin, \cos)$ phase.
- **Ghosts**: distance, relative altitude, `vx`, `vy`, float amplitude/speed, $(\sin, \cos)$ wave phase, and patrol bounds.
- **Rotating Lasers**: distance, $(\sin \theta, \cos \theta)$, blocking flag, eye `(dx, dy)`, `angular_vel`, length, and sweep amplitude.
- **Wisps**: distance, direction, grounded flag, charging, caster `(dx, dy)`, cast progress, ticks to cast, next-high flag, `dy`, `vx`, `vy`, and roll ticks.
- **Grave Monsters**: distance, relative elevation, `vx`, `vy`, grounded, sprinting, state, trigger distance, sentry, pause progress, speed ratio, and pit distance.
- **Goal & Items**: distance to next garlic item and remaining level distance.


In [ ]:
import math
from typing import Optional
import numpy as np
from core.base_env import RawState, Action

class ContinuousStateExtractor:
    """
    Extracts a normalized, stationary continuous feature vector from RawState.
    
    Transforms the continuous physical and sensory measurements into a fixed-length
    1D numpy array suitable for Deep Q-Networks (DQN).
    
    Synchronized with the latest game_frontend_v4 & Section 7 DQN-ready dynamics:
      - Player kinematics & posture (6)
      - Tombstone obstacles: dist, height, dy, width (4)
      - Pit & periodic hellfire: dist, width, state, cycle, countdown, sin, cos (7)
      - 2D Ghost patrol & wave kinematics: dist, rel_y, vx, vy, amp, speed, sin, cos, bounds (10)
      - Rotating Laser hazard: dist, sin, cos, blocking, dx, dy, angular_vel, length, sweep_amp (9)
      - Will-o'-the-wisp & WispGhost casting: dist, dir, grounded, charging, dx, dy, progress, ticks, next_high, wisp_dy, vx, vy, roll_ticks (13)
      - Grave monsters (Quỷ Nhập Tràng & Cương Thi): dist, rel_y, vx, grounded, sprinting, vy, state, trigger, sentry, pause_progress, speed_ratio, dist_to_pit (12)
      - Warmth item & goal distance (2)

    Feature vector dimension: 63
    """

    STATE_DIM: int = 63

    def __init__(
        self,
        canvas_height: float = 400.0,
        max_threat_distance: float = 200.0,
        max_level_length: float = 8000.0,
    ) -> None:
        self.canvas_height = canvas_height
        self.max_threat_distance = max_threat_distance
        self.max_level_length = max_level_length

    def extract(self, raw: RawState) -> np.ndarray:
        player_y = getattr(raw, "player_y", getattr(raw, "mario_y", 288.0))
        vy = getattr(raw, "velocity_y", 0.0)
        vx = getattr(raw, "velocity_x", 0.0)

        # 1. Player Kinematics & Posture (6 features)
        norm_y = np.clip(player_y / self.canvas_height, 0.0, 1.0)
        norm_vx = np.clip(vx / 7.0, -1.0, 1.0)
        norm_vy = np.clip(vy / 15.0, -1.0, 1.0)
        grounded = 1.0 if getattr(raw, "is_grounded", False) else 0.0
        double_jump = 1.0 if getattr(raw, "can_double_jump", False) else 0.0
        crouch = 1.0 if getattr(raw, "is_crouching", False) else 0.0

        # 2. Tombstone Obstacles (4 features)
        dist_obs = getattr(raw, "dist_to_obstacle", -1.0)
        norm_dist_obs = (
            np.clip(dist_obs / self.max_threat_distance, 0.0, 1.0)
            if dist_obs >= 0.0
            else 1.0
        )
        norm_height_obs = np.clip(getattr(raw, "obstacle_height", 0.0) / 100.0, 0.0, 1.0)
        norm_obs_dy = (
            np.clip(getattr(raw, "obstacle_dy", 0.0) / self.max_threat_distance, -1.0, 1.0)
            if dist_obs >= 0.0
            else 0.0
        )
        norm_obs_width = np.clip(getattr(raw, "obstacle_width", 0.0) / 100.0, 0.0, 1.0)

        # 3. Grave Pits & Periodic Hellfire (7 features)
        dist_pit = getattr(raw, "dist_to_pit", -1.0)
        norm_dist_pit = (
            np.clip(dist_pit / self.max_threat_distance, 0.0, 1.0)
            if dist_pit >= 0.0
            else 1.0
        )
        pit_width = getattr(raw, "pit_width", 0.0)
        norm_pit_width = np.clip(pit_width / 150.0, 0.0, 1.0)
        norm_fire_state = np.clip(getattr(raw, "pit_fire_state", 0) / 2.0, 0.0, 1.0)
        pit_fire_cycle = np.clip(getattr(raw, "pit_fire_cycle", 0.0), 0.0, 1.0)
        norm_fire_countdown = np.clip(getattr(raw, "pit_fire_countdown", 0.0) / 100.0, 0.0, 1.0)
        pit_fire_sin = float(np.clip(getattr(raw, "pit_fire_sin", 0.0), -1.0, 1.0))
        pit_fire_cos = float(np.clip(getattr(raw, "pit_fire_cos", 1.0), -1.0, 1.0))

        # 4. Ghost Proximity & Wave Kinematics (10 features)
        dist_ghost = getattr(raw, "dist_to_ghost", -1.0)
        norm_dist_ghost = (
            np.clip(dist_ghost / self.max_threat_distance, 0.0, 1.0)
            if dist_ghost >= 0.0
            else 1.0
        )
        ghost_y = getattr(raw, "ghost_y", 0.0)
        if dist_ghost >= 0.0 and ghost_y > 0.0:
            rel_ghost_y = np.clip((ghost_y - player_y) / self.max_threat_distance, -1.0, 1.0)
        elif dist_ghost >= 0.0:
            rel_ghost_y = np.clip(getattr(raw, "ghost_dy", 0.0) / self.max_threat_distance, -1.0, 1.0)
        else:
            rel_ghost_y = 0.0
        norm_ghost_vx = np.clip(getattr(raw, "ghost_vx", 0.0) / 5.0, -1.0, 1.0)
        norm_ghost_vy = np.clip(getattr(raw, "ghost_vy", 0.0) / 5.0, -1.0, 1.0)
        norm_ghost_amp = np.clip(getattr(raw, "ghost_float_amplitude", 0.0) / 100.0, 0.0, 1.0)
        norm_ghost_float_speed = np.clip(getattr(raw, "ghost_float_speed", 0.0) / 0.5, -1.0, 1.0)
        ghost_phase_sin = float(np.clip(getattr(raw, "ghost_phase_sin", 0.0), -1.0, 1.0))
        ghost_phase_cos = float(np.clip(getattr(raw, "ghost_phase_cos", 1.0), -1.0, 1.0))
        norm_ghost_left_bound = np.clip(getattr(raw, "ghost_dist_to_left_bound", 0.0) / 250.0, 0.0, 1.0)
        norm_ghost_right_bound = np.clip(getattr(raw, "ghost_dist_to_right_bound", 0.0) / 250.0, 0.0, 1.0)

        # 5. Rotating Laser Hazard (9 features)
        dist_laser = getattr(raw, "dist_to_laser", -1.0)
        norm_dist_laser = (
            np.clip(dist_laser / self.max_threat_distance, 0.0, 1.0)
            if dist_laser >= 0.0
            else 1.0
        )
        laser_angle = getattr(raw, "laser_angle", -1.0)
        raw_laser_sin = getattr(raw, "laser_angle_sin", None)
        raw_laser_cos = getattr(raw, "laser_angle_cos", None)
        if dist_laser >= 0.0 and raw_laser_sin is not None and raw_laser_cos is not None and (raw_laser_sin != 0.0 or raw_laser_cos != 1.0 or laser_angle == 0.0):
            laser_sin = float(np.clip(raw_laser_sin, -1.0, 1.0))
            laser_cos = float(np.clip(raw_laser_cos, -1.0, 1.0))
        elif dist_laser >= 0.0 and laser_angle >= 0.0:
            laser_sin = float(math.sin(laser_angle))
            laser_cos = float(math.cos(laser_angle))
        else:
            laser_sin = 0.0
            laser_cos = 0.0
        laser_is_blocking = 1.0 if getattr(raw, "laser_is_blocking", False) else 0.0
        norm_laser_ghost_dx = np.clip(getattr(raw, "laser_ghost_dx", 0.0) / self.max_threat_distance, -1.0, 1.0)
        norm_laser_ghost_dy = np.clip(getattr(raw, "laser_ghost_dy", 0.0) / self.max_threat_distance, -1.0, 1.0)
        norm_laser_angular_vel = np.clip(
            getattr(raw, "laser_angular_vel", getattr(raw, "laser_rot_speed", 0.0)) / 0.2,
            -1.0,
            1.0,
        )
        norm_laser_length = np.clip(getattr(raw, "laser_length", 0.0) / 200.0, 0.0, 1.0)
        norm_laser_sweep_amp = np.clip(getattr(raw, "laser_sweep_amplitude", 0.0) / math.pi, 0.0, 1.0)

        # 6. Will-o'-the-Wisp Projectile & Casting (13 features)
        dist_wisp = getattr(raw, "dist_to_wisp", -1.0)
        norm_dist_wisp = (
            np.clip(dist_wisp / self.max_threat_distance, 0.0, 1.0)
            if dist_wisp >= 0.0
            else 1.0
        )
        norm_wisp_dir = float(np.clip(getattr(raw, "wisp_dir", 0), -1, 1))
        wisp_is_grounded = 1.0 if getattr(raw, "wisp_is_grounded", False) else 0.0
        wisp_charging = 1.0 if (
            getattr(raw, "wisp_charging", False) or getattr(raw, "wisp_ghost_charging", False)
        ) else 0.0
        norm_wisp_ghost_dx = np.clip(getattr(raw, "wisp_ghost_dx", 0.0) / self.max_threat_distance, -1.0, 1.0)
        norm_wisp_ghost_dy = np.clip(getattr(raw, "wisp_ghost_dy", 0.0) / self.max_threat_distance, -1.0, 1.0)
        wisp_cast_progress = float(np.clip(getattr(raw, "wisp_cast_progress", 0.0), 0.0, 1.0))
        norm_wisp_ticks_to_cast = np.clip(getattr(raw, "wisp_ticks_to_cast", 0.0) / 200.0, 0.0, 1.0)
        wisp_next_is_high = 1.0 if getattr(raw, "wisp_next_is_high", False) else 0.0
        norm_wisp_dy = np.clip(getattr(raw, "wisp_dy", 0.0) / self.max_threat_distance, -1.0, 1.0)
        norm_wisp_vx = np.clip(getattr(raw, "wisp_vx", 0.0) / 8.0, -1.0, 1.0)
        norm_wisp_vy = np.clip(getattr(raw, "wisp_vy", 0.0) / 8.0, -1.0, 1.0)
        norm_wisp_roll_ticks = np.clip(getattr(raw, "wisp_roll_ticks_left", 0) / 100.0, 0.0, 1.0)

        # 7. Grave Monsters (Quỷ Nhập Tràng & Cương Thi) (12 features)
        dist_monster = getattr(raw, "dist_to_monster", -1.0)
        norm_dist_monster = (
            np.clip(dist_monster / self.max_threat_distance, 0.0, 1.0)
            if dist_monster >= 0.0
            else 1.0
        )
        monster_y = getattr(raw, "monster_y", 0.0)
        if dist_monster >= 0.0 and monster_y > 0.0:
            rel_monster_y = np.clip((monster_y - player_y) / self.max_threat_distance, -1.0, 1.0)
        elif dist_monster >= 0.0:
            rel_monster_y = np.clip(getattr(raw, "monster_dy", 0.0) / self.max_threat_distance, -1.0, 1.0)
        else:
            rel_monster_y = 0.0
        monster_vx = getattr(raw, "monster_vx", 0.0)
        norm_monster_vx = np.clip(monster_vx / 5.0, -1.0, 1.0)
        norm_monster_vy = np.clip(getattr(raw, "monster_vy", 0.0) / 15.0, -1.0, 1.0)
        monster_is_grounded = 1.0 if getattr(raw, "monster_is_grounded", True) else 0.0
        monster_type = getattr(raw, "monster_type", None)
        monster_is_sprinting = 1.0 if (
            monster_type in ("quy_nhap_trang", 1) or monster_vx < -2.0
        ) else 0.0
        norm_monster_state = np.clip(getattr(raw, "monster_state", 0) / 2.0, 0.0, 1.0)
        dist_trig = getattr(raw, "dist_to_monster_trigger", -1.0)
        norm_monster_trigger = (
            np.clip(dist_trig / self.max_threat_distance, 0.0, 1.0)
            if dist_trig >= 0.0
            else 1.0
        )
        monster_is_sentry = 1.0 if getattr(raw, "monster_is_sentry", False) else 0.0
        monster_pause_progress = float(np.clip(getattr(raw, "monster_pause_progress", 0.0), 0.0, 1.0))
        monster_speed_ratio = float(np.clip(getattr(raw, "monster_speed_ratio", 0.0), 0.0, 1.0))
        m_pit = getattr(raw, "monster_dist_to_pit", -1.0)
        norm_monster_dist_to_pit = (
            np.clip(m_pit / self.max_threat_distance, 0.0, 1.0)
            if m_pit >= 0.0
            else 1.0
        )

        # 8. Warmth Garlic Item & Level Goal (2 features)
        dist_item = getattr(raw, "dist_to_item", -1.0)
        norm_dist_item = (
            np.clip(dist_item / self.max_threat_distance, 0.0, 1.0)
            if dist_item >= 0.0
            else 1.0
        )
        dist_goal = getattr(raw, "dist_to_goal", self.max_level_length)
        norm_dist_goal = np.clip(dist_goal / self.max_level_length, 0.0, 1.0)

        return np.array(
            [
                # 1. Kinematics (6)
                norm_y,
                norm_vx,
                norm_vy,
                grounded,
                double_jump,
                crouch,
                # 2. Obstacles (4)
                norm_dist_obs,
                norm_height_obs,
                norm_obs_dy,
                norm_obs_width,
                # 3. Pits (7)
                norm_dist_pit,
                norm_pit_width,
                norm_fire_state,
                pit_fire_cycle,
                norm_fire_countdown,
                pit_fire_sin,
                pit_fire_cos,
                # 4. Ghosts (10)
                norm_dist_ghost,
                rel_ghost_y,
                norm_ghost_vx,
                norm_ghost_vy,
                norm_ghost_amp,
                norm_ghost_float_speed,
                ghost_phase_sin,
                ghost_phase_cos,
                norm_ghost_left_bound,
                norm_ghost_right_bound,
                # 5. Lasers (9)
                norm_dist_laser,
                laser_sin,
                laser_cos,
                laser_is_blocking,
                norm_laser_ghost_dx,
                norm_laser_ghost_dy,
                norm_laser_angular_vel,
                norm_laser_length,
                norm_laser_sweep_amp,
                # 6. Wisps (13)
                norm_dist_wisp,
                norm_wisp_dir,
                wisp_is_grounded,
                wisp_charging,
                norm_wisp_ghost_dx,
                norm_wisp_ghost_dy,
                wisp_cast_progress,
                norm_wisp_ticks_to_cast,
                wisp_next_is_high,
                norm_wisp_dy,
                norm_wisp_vx,
                norm_wisp_vy,
                norm_wisp_roll_ticks,
                # 7. Monsters (12)
                norm_dist_monster,
                rel_monster_y,
                norm_monster_vx,
                monster_is_grounded,
                monster_is_sprinting,
                norm_monster_vy,
                norm_monster_state,
                norm_monster_trigger,
                monster_is_sentry,
                monster_pause_progress,
                monster_speed_ratio,
                norm_monster_dist_to_pit,
                # 8. Items & Goal (2)
                norm_dist_item,
                norm_dist_goal,
            ],
            dtype=np.float32,
        )

print(f"[OK] ContinuousStateExtractor initialized. State dimension: {ContinuousStateExtractor.STATE_DIM}")


## 4. Dueling Double Deep Q-Network (D3QN) Architecture
The network decouples into a state-value stream $V(s)$ and an advantage stream $A(s, a)$:

$$Q(s, a) = V(s) + \left( A(s, a) - \frac{1}{|\mathcal{A}|} \sum_{a' \in \mathcal{A}} A(s, a') \right)$$


In [ ]:
import torch
import torch.nn as nn

class DuelingDQN(nn.Module):
    """
    Dueling Deep Q-Network with Layer Normalization and Separated V(s) / A(s, a) Streams.
    """
    def __init__(self, state_dim: int = 63, num_actions: int = 6, hidden_dim: int = 128):
        super().__init__()
        self.state_dim = state_dim
        self.num_actions = num_actions

        # Shared feature trunk
        self.feature_net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        # Value Stream V(s)
        self.value_stream = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

        # Advantage Stream A(s, a)
        self.advantage_stream = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_actions),
        )

    def forward(self, state: torch.Tensor) -> torch.Tensor:
        features = self.feature_net(state)
        values = self.value_stream(features)
        advantages = self.advantage_stream(features)
        # Aggregation with zero-mean advantage centering
        return values + (advantages - advantages.mean(dim=-1, keepdim=True))

sample_net = DuelingDQN().to(device)
dummy_input = torch.randn(2, ContinuousStateExtractor.STATE_DIM).to(device)
out = sample_net(dummy_input)
print(f"[OK] DuelingDQN built successfully! Test output shape: {out.shape}")


## 5. Replay Buffer & DQNAgent
Includes Double Q-learning target computation, soft Polyak target update, and Huber Loss.


In [ ]:
from collections import deque
import random
from typing import Tuple
import torch.optim as optim

class ReplayBuffer:
    def __init__(self, capacity: int = 100_000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state: np.ndarray, action: int, reward: float, next_state: np.ndarray, done: bool) -> None:
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size: int, device: torch.device):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.as_tensor(np.array(states), dtype=torch.float32, device=device),
            torch.as_tensor(np.array(actions), dtype=torch.int64, device=device).unsqueeze(1),
            torch.as_tensor(np.array(rewards), dtype=torch.float32, device=device).unsqueeze(1),
            torch.as_tensor(np.array(next_states), dtype=torch.float32, device=device),
            torch.as_tensor(np.array(dones), dtype=torch.float32, device=device).unsqueeze(1),
        )

    def __len__(self) -> int:
        return len(self.buffer)


class DQNAgent:
    def __init__(
        self,
        state_dim: int = 63,
        num_actions: int = 6,
        learning_rate: float = 3e-4,
        gamma: float = 0.99,
        tau: float = 0.005,
        epsilon: float = 1.0,
        epsilon_min: float = 0.01,
        epsilon_decay: float = 0.995,
        buffer_capacity: int = 100_000,
        batch_size: int = 64,
        warmup_steps: int = 500,
        device: torch.device = device,
    ):
        self.state_dim = state_dim
        self.num_actions = num_actions
        self.gamma = gamma
        self.tau = tau
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.batch_size = batch_size
        self.warmup_steps = warmup_steps
        self.device = device
        self.training = True

        self.policy_net = DuelingDQN(state_dim, num_actions).to(self.device)
        self.target_net = DuelingDQN(state_dim, num_actions).to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=learning_rate)
        self.loss_fn = nn.SmoothL1Loss()
        self.replay_buffer = ReplayBuffer(capacity=buffer_capacity)

    def select_action(self, state: np.ndarray) -> int:
        if self.training and random.random() < self.epsilon:
            return random.randrange(self.num_actions)

        state_t = torch.as_tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
        with torch.no_grad():
            q_values = self.policy_net(state_t)
            return int(q_values.argmax(dim=1).item())

    def update(self, state: np.ndarray, action: int, reward: float, next_state: np.ndarray, done: bool) -> float:
        self.replay_buffer.push(state, action, reward, next_state, done)

        if not self.training or len(self.replay_buffer) < self.warmup_steps:
            return 0.0

        states, actions, rewards, next_states, dones = self.replay_buffer.sample(self.batch_size, self.device)

        # Double DQN: Policy net selects best action, Target net evaluates value
        with torch.no_grad():
            best_actions = self.policy_net(next_states).argmax(dim=1, keepdim=True)
            next_q = self.target_net(next_states).gather(1, best_actions)
            target_q = rewards + (1.0 - dones) * self.gamma * next_q

        current_q = self.policy_net(states).gather(1, actions)
        loss = self.loss_fn(current_q, target_q)

        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), max_norm=10.0)
        self.optimizer.step()

        # Polyak soft update
        with torch.no_grad():
            for p, tp in zip(self.policy_net.parameters(), self.target_net.parameters()):
                tp.data.copy_(self.tau * p.data + (1.0 - self.tau) * tp.data)

        return float(loss.item())

    def decay_epsilon(self) -> float:
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
        return self.epsilon

    def save(self, filepath: str) -> None:
        Path(filepath).parent.mkdir(parents=True, exist_ok=True)
        torch.save({
            "policy_net": self.policy_net.state_dict(),
            "target_net": self.target_net.state_dict(),
            "epsilon": self.epsilon,
        }, filepath)

    def load(self, filepath: str) -> None:
        ckpt = torch.load(filepath, map_location=self.device)
        self.policy_net.load_state_dict(ckpt["policy_net"])
        self.target_net.load_state_dict(ckpt["target_net"])
        self.epsilon = ckpt.get("epsilon", self.epsilon_min)

print("[OK] ReplayBuffer and DQNAgent defined successfully!")


## 6. Curriculum Training Across Levels 1 to 8
Now we train the DQN agent. The curriculum trains through Level 1 to Level 8 progressively.
Checkpoint weights are inherited between stages, enabling smooth skill transfer.


In [ ]:
import time
import pandas as pd
from env import create_env
from rewards.balanced_reward import BalancedRewardStrategy

def train_curriculum(
    levels=[1, 2, 3, 4, 5, 6, 7, 8],
    episodes_per_level=150,
    lr=3e-4,
    batch_size=64,
    frame_skip=1,
    checkpoint_dir="checkpoints/dqn_colab",
    pretrained_path=None,
):
    ckpt_dir = Path(checkpoint_dir)
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    extractor = ContinuousStateExtractor()
    reward_fn = BalancedRewardStrategy()
    agent = DQNAgent(
        state_dim=ContinuousStateExtractor.STATE_DIM,
        num_actions=6,
        learning_rate=lr,
        batch_size=batch_size,
        device=device,
    )

    if pretrained_path is not None and Path(pretrained_path).exists():
        print(f"[PRETRAINED] Resuming from checkpoint: {pretrained_path}")
        agent.load(pretrained_path)
        agent.epsilon = max(0.25, agent.epsilon)

    history = []
    global_ep = 0
    total_episodes = len(levels) * episodes_per_level
    start_time = time.time()

    print(f"================== STARTING DQN CURRICULUM ({len(levels)} Levels) ==================")

    for lvl in levels:
        env = create_env(level_id=lvl)
        lvl_wins = []
        best_win_rate = 0.0

        # Maintain minimum exploration when entering harder levels
        if lvl > levels[0] and agent.epsilon < 0.20:
            agent.epsilon = 0.20

        print(f"\n---> [LEVEL {lvl}] Starting {episodes_per_level} episodes (eps={agent.epsilon:.2f})")

        for ep in range(1, episodes_per_level + 1):
            global_ep += 1
            if hasattr(reward_fn, "reset"):
                reward_fn.reset()
            raw_state = env.reset()
            state_vec = extractor.extract(raw_state)
            ep_reward = 0.0
            ep_loss = 0.0
            steps = 0
            loss_count = 0
            done = False

            while not done:
                action = agent.select_action(state_vec)

                # Frame skipping
                skip_reward = 0.0
                for _ in range(frame_skip):
                    next_raw, events, done = env.step(action)
                    step_r = reward_fn.calculate_reward(raw_state, action, next_raw, events, done)
                    skip_reward += step_r
                    raw_state = next_raw
                    steps += 1
                    if done:
                        break

                next_vec = extractor.extract(next_raw)
                loss = agent.update(state_vec, action, skip_reward, next_vec, done)
                if loss > 0.0:
                    ep_loss += loss
                    loss_count += 1

                ep_reward += skip_reward
                state_vec = next_vec

            agent.decay_epsilon()
            win = 1 if getattr(events, "reached_goal", False) else 0
            lvl_wins.append(win)
            rolling_50 = float(np.mean(lvl_wins[-50:])) * 100.0
            avg_loss = ep_loss / max(1, loss_count)

            history.append({
                "global_episode": global_ep,
                "level_id": lvl,
                "level_episode": ep,
                "reward": ep_reward,
                "steps": steps,
                "win": win,
                "rolling_win_rate_50": rolling_50,
                "loss": avg_loss,
                "epsilon": agent.epsilon,
            })

            if len(lvl_wins) >= 20 and rolling_50 > best_win_rate:
                best_win_rate = rolling_50
                agent.save(str(ckpt_dir / f"dqn_level_{lvl}_best.pt"))

            if ep == 1 or ep % 25 == 0 or ep == episodes_per_level:
                elapsed = time.time() - start_time
                print(
                    f"[{global_ep:4d}/{total_episodes:4d}] Lvl {lvl} (Ep {ep:3d}/{episodes_per_level:3d}) | "
                    f"Win50: {rolling_50:5.1f}% | R: {ep_reward:+7.1f} | Loss: {avg_loss:.4f} | "
                    f"eps: {agent.epsilon:.3f} | Elapsed: {int(elapsed)}s"
                )

        env.close()

    agent.save(str(ckpt_dir / "dqn_final.pt"))
    df_metrics = pd.DataFrame(history)
    df_metrics.to_csv(ckpt_dir / "training_metrics.csv", index=False)
    print(f"\n[COMPLETE] Training finished! Model and metrics saved to {ckpt_dir}")
    return agent, df_metrics


## 7. Run DQN Training
*Tip*: In Colab / Kaggle, training Levels 1–4 takes ~2–3 minutes on GPU. To train all 8 levels, set `levels=[1, 2, 3, 4, 5, 6, 7, 8]` and `episodes_per_level=150`.


In [ ]:
# Configure and run training (Adjust levels and episodes as needed)
TRAIN_LEVELS = [1, 2, 3, 4]  # Expand to [1, 2, 3, 4, 5, 6, 7, 8] for the full marathon!
EPISODES_PER_LEVEL = 100

trained_agent, df_metrics = train_curriculum(
    levels=TRAIN_LEVELS,
    episodes_per_level=EPISODES_PER_LEVEL,
    lr=3e-4,
    batch_size=64,
    frame_skip=1,
)


## 8. Training Telemetry & Learning Curves
Plot the training metrics: Episodic Return, Rolling Win Rate %, TD Loss, and Steps per Episode.


In [ ]:
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Episodic Reward
axes[0, 0].plot(df_metrics["global_episode"], df_metrics["reward"], alpha=0.3, color="royalblue", label="Raw Reward")
axes[0, 0].plot(df_metrics["global_episode"], df_metrics["reward"].rolling(20, min_periods=1).mean(), color="navy", lw=2, label="Moving Avg (20)")
axes[0, 0].set_title("Episodic Return", fontsize=13, fontweight="bold")
axes[0, 0].set_xlabel("Global Episode")
axes[0, 0].set_ylabel("Reward")
axes[0, 0].legend()

# 2. Rolling Win Rate %
axes[0, 1].plot(df_metrics["global_episode"], df_metrics["rolling_win_rate_50"], color="forestgreen", lw=2.5)
axes[0, 1].set_title("50-Episode Rolling Win Rate (%)", fontsize=13, fontweight="bold")
axes[0, 1].set_xlabel("Global Episode")
axes[0, 1].set_ylabel("Win Rate (%)")
axes[0, 1].set_ylim(-5, 105)

# 3. TD Loss
axes[1, 0].plot(df_metrics["global_episode"], df_metrics["loss"], color="crimson", alpha=0.7)
axes[1, 0].set_title("Temporal Difference Huber Loss", fontsize=13, fontweight="bold")
axes[1, 0].set_xlabel("Global Episode")
axes[1, 0].set_ylabel("Loss")

# 4. Steps per Episode
axes[1, 1].plot(df_metrics["global_episode"], df_metrics["steps"], color="darkorange", alpha=0.6)
axes[1, 1].set_title("Steps per Episode", fontsize=13, fontweight="bold")
axes[1, 1].set_xlabel("Global Episode")
axes[1, 1].set_ylabel("Steps")

plt.tight_layout()
plt.savefig("checkpoints/dqn_colab/learning_curves.png", dpi=150)
plt.show()
print("[OK] Plots generated and saved to checkpoints/dqn_colab/learning_curves.png")


## 9. Greedy Evaluation Benchmark across Levels
Evaluate the trained DQN policy under pure exploitation ($\epsilon = 0.0$) across evaluated levels.


In [ ]:
def evaluate_agent(agent, levels, num_trials=10):
    agent.training = False
    agent.epsilon = 0.0
    extractor = ContinuousStateExtractor()
    reward_fn = BalancedRewardStrategy()
    eval_results = []

    print(f"================ EVALUATION MATRIX (Epsilon = 0.0) ================")
    print(f"{'Level':<8} {'Win Rate':<12} {'Avg Reward':<14} {'Avg Steps':<12} {'Status'}")
    print("-" * 55)

    for lvl in levels:
        env = create_env(level_id=lvl)
        wins = 0
        rewards = []
        steps_list = []

        for _ in range(num_trials):
            if hasattr(reward_fn, "reset"):
                reward_fn.reset()
            raw_state = env.reset()
            state_vec = extractor.extract(raw_state)
            tot_r = 0.0
            steps = 0
            done = False

            while not done:
                action = agent.select_action(state_vec)
                next_raw, events, done = env.step(action)
                r = reward_fn.calculate_reward(raw_state, action, next_raw, events, done)
                tot_r += r
                steps += 1
                state_vec = extractor.extract(next_raw)
                raw_state = next_raw

            if getattr(events, "reached_goal", False):
                wins += 1
            rewards.append(tot_r)
            steps_list.append(steps)

        env.close()
        win_pct = (wins / num_trials) * 100.0
        avg_r = np.mean(rewards)
        avg_s = np.mean(steps_list)
        status = "PASSED" if win_pct >= 80.0 else "IN PROGRESS"

        eval_results.append({
            "Level": lvl,
            "Win_Rate_Pct": win_pct,
            "Avg_Reward": avg_r,
            "Avg_Steps": avg_s,
            "Status": status,
        })
        print(f"Level {lvl:<3} {win_pct:6.1f}%       {avg_r:+8.1f}       {avg_s:6.1f}       {status}")

    df_eval = pd.DataFrame(eval_results)
    return df_eval

df_eval = evaluate_agent(trained_agent, levels=TRAIN_LEVELS, num_trials=10)


## 10. Watch the Agent Play (Animated GIF Generator)
Since headless cloud environments have no active window display, we render Pygame surfaces directly to an off-screen buffer and save an animated GIF to watch inside the notebook!


In [ ]:
import imageio
from IPython.display import Image, display

def record_gameplay_gif(agent, level_id=1, output_path="gameplay.gif", max_frames=600):
    """
    Simulates one episode of the trained agent and renders an animated GIF.
    """
    env = create_env(level_id=level_id)
    extractor = ContinuousStateExtractor()
    frames = []

    raw_state = env.reset()
    state_vec = extractor.extract(raw_state)
    done = False
    frame_count = 0

    agent.training = False
    agent.epsilon = 0.0

    print(f"[RECORDING] Capturing Level {level_id} gameplay frames...")

    while not done and frame_count < max_frames:
        action = agent.select_action(state_vec)
        next_raw, events, done = env.step(action)
        state_vec = extractor.extract(next_raw)

        # Render off-screen
        env.render()
        if env._renderer is not None and env._renderer._screen is not None:
            # Capture RGB surface array from Pygame
            surf = env._renderer._screen
            img_data = pygame.surfarray.array3d(surf)
            # Transpose axes from (W, H, C) to (H, W, C)
            img_data = np.transpose(img_data, (1, 0, 2))
            frames.append(img_data)

        frame_count += 1

    env.close()

    # Downsample slightly for fast rendering and small GIF size
    step_skip = 2
    sampled_frames = frames[::step_skip]
    imageio.mimsave(output_path, sampled_frames, fps=25)
    print(f"[OK] Gameplay GIF saved to {output_path} ({len(sampled_frames)} frames)")
    return output_path

gif_path = record_gameplay_gif(trained_agent, level_id=1, output_path="gameplay_level_1.gif")
display(Image(filename=gif_path))


## 11. Download Trained Models & Artifacts
Compress checkpoints and metrics into a zip archive and download to your machine.


In [ ]:
import shutil

# Create a zip bundle of trained models and plots
shutil.make_archive("mario_dqn_checkpoints", "zip", "checkpoints/dqn_colab")
print("[OK] Created mario_dqn_checkpoints.zip")

# Trigger automatic browser download if in Google Colab
if is_colab:
    try:
        from google.colab import files
        files.download("mario_dqn_checkpoints.zip")
        print("[INFO] Download triggered in Colab!")
    except Exception as e:
        print(f"[NOTE] Manual download available: {e}")
elif is_kaggle:
    print("[INFO] Checkpoints available in Kaggle Output directory: /kaggle/working/mario_dqn_checkpoints.zip")
